# Day 68 — Cloud Deployment for ML
### AWS S3, EC2, Lambda, SageMaker — concepts, cost estimation, deployment patterns

## 1. Setup

In [1]:
import pandas as pd
import numpy as np

print("Setup ready")
print("Note: AWS demos require boto3 + AWS credentials")
print("Today covers concepts, patterns, and cost estimation")

Setup ready
Note: AWS demos require boto3 + AWS credentials
Today covers concepts, patterns, and cost estimation


## 2. Cloud Providers for ML — Choosing the Right One

In [2]:
providers = pd.DataFrame([
    {
        "provider": "AWS",
        "ml_services": "SageMaker, Bedrock, Rekognition, Comprehend",
        "strengths": "Largest ecosystem, most mature ML tooling, widest region coverage",
        "weaknesses": "Complex pricing, steep learning curve",
        "best_for": "Enterprise ML, most job postings require AWS knowledge"
    },
    {
        "provider": "GCP",
        "ml_services": "Vertex AI, AutoML, BigQuery ML, Gemini API",
        "strengths": "Best data/analytics integration (BigQuery), strong TPU support",
        "weaknesses": "Smaller ecosystem than AWS",
        "best_for": "Data-heavy ML, NLP via Gemini, Kaggle integration"
    },
    {
        "provider": "Azure",
        "ml_services": "Azure ML, OpenAI Service, Cognitive Services",
        "strengths": "Best for Microsoft/enterprise stack, OpenAI integration",
        "weaknesses": "Less ML-native than AWS/GCP",
        "best_for": "Enterprise with existing Microsoft contracts"
    },
    {
        "provider": "AWS (focus today)",
        "ml_services": "S3 + EC2 + Lambda + SageMaker",
        "strengths": "Most common in DS/AI job descriptions",
        "weaknesses": "--",
        "best_for": "Industry-standard -- knowing AWS unlocks the most opportunities"
    },
])

providers[["provider", "ml_services", "best_for"]]

,provider,ml_services,best_for
0,AWS,"SageMaker, Bedrock, Rekognition, Comprehend","Enterprise ML, most job postings require AWS k..."
1,GCP,"Vertex AI, AutoML, BigQuery ML, Gemini API","Data-heavy ML, NLP via Gemini, Kaggle integration"
2,Azure,"Azure ML, OpenAI Service, Cognitive Services",Enterprise with existing Microsoft contracts
3,AWS (focus today),S3 + EC2 + Lambda + SageMaker,Industry-standard -- knowing AWS unlocks the m...


## 3. AWS S3 — Data Storage for ML

In [3]:
s3_patterns = """
AWS S3 (Simple Storage Service) -- Object Storage for ML
=========================================================

WHAT S3 IS:
  Infinitely scalable object storage -- store any file (datasets, models,
  artifacts, logs) at any scale. Accessed via URL or boto3 SDK.

KEY ML USE CASES:
  1. Training data storage    -- store raw datasets, versioned by prefix
  2. Model artifact storage   -- save trained .pkl / .pt / .onnx files
  3. Feature store            -- store precomputed features as parquet
  4. Prediction logging       -- append prediction logs for monitoring
  5. MLflow artifact backend  -- MLflow can use S3 as its artifact store

S3 STRUCTURE:
  s3://my-bucket/
    data/
      raw/          train.csv, test.csv
      processed/    features_v1.parquet
    models/
      v1.0/         model.pkl, scaler.pkl
      v2.0/         model.pkl, scaler.pkl
    logs/
      predictions/  2026-07-14.jsonl

BOTO3 PATTERN (Python SDK):
  import boto3
  s3 = boto3.client('s3')

  # Upload model to S3
  s3.upload_file('model.pkl', 'my-bucket', 'models/v1.0/model.pkl')

  # Download model from S3
  s3.download_file('my-bucket', 'models/v1.0/model.pkl', 'model.pkl')

  # List objects with prefix
  response = s3.list_objects_v2(Bucket='my-bucket', Prefix='models/')
  for obj in response['Contents']:
      print(obj['Key'], obj['Size'])

S3 PRICING (us-east-1):
  Storage:  $0.023 per GB/month
  Requests: $0.0004 per 1,000 GET requests
  Transfer: $0.09 per GB out (first GB free)

  Example: 100GB dataset + 10,000 training reads/month = ~$2.30 + $0.004 = ~$2.31
"""
print(s3_patterns)


AWS S3 (Simple Storage Service) -- Object Storage for ML

WHAT S3 IS:
  Infinitely scalable object storage -- store any file (datasets, models,
  artifacts, logs) at any scale. Accessed via URL or boto3 SDK.

KEY ML USE CASES:
  1. Training data storage    -- store raw datasets, versioned by prefix
  2. Model artifact storage   -- save trained .pkl / .pt / .onnx files
  3. Feature store            -- store precomputed features as parquet
  4. Prediction logging       -- append prediction logs for monitoring
  5. MLflow artifact backend  -- MLflow can use S3 as its artifact store

S3 STRUCTURE:
  s3://my-bucket/
    data/
      raw/          train.csv, test.csv
      processed/    features_v1.parquet
    models/
      v1.0/         model.pkl, scaler.pkl
      v2.0/         model.pkl, scaler.pkl
    logs/
      predictions/  2026-07-14.jsonl

BOTO3 PATTERN (Python SDK):
  import boto3
  s3 = boto3.client('s3')

  # Upload model to S3
  s3.upload_file('model.pkl', 'my-bucket', 'models/v1

## 4. EC2 vs Lambda vs SageMaker — When to Use Which

In [4]:
serving_options = pd.DataFrame([
    {
        "service": "EC2 (Virtual Machine)",
        "type": "Always-on server",
        "cold_start": "None (always running)",
        "max_runtime": "Unlimited",
        "cost_model": "Per hour (whether idle or not)",
        "best_for": "High-traffic APIs, models needing GPU, full control",
        "example_cost": "t3.medium: ~$30/month"
    },
    {
        "service": "Lambda (Serverless)",
        "type": "Event-driven, scales to zero",
        "cold_start": "100ms - 2s",
        "max_runtime": "15 minutes max",
        "cost_model": "Per invocation + duration (free tier: 1M requests/month)",
        "best_for": "Low-traffic APIs, lightweight models (<250MB), event triggers",
        "example_cost": "1M requests/month: ~$0.20"
    },
    {
        "service": "SageMaker Endpoint",
        "type": "Managed ML inference",
        "cold_start": "None (always-on) or minutes (async)",
        "max_runtime": "Unlimited",
        "cost_model": "Per hour per instance type",
        "best_for": "Production ML serving, auto-scaling, A/B testing built-in",
        "example_cost": "ml.t2.medium: ~$50/month"
    },
    {
        "service": "SageMaker Serverless",
        "type": "Serverless ML inference",
        "cold_start": "1-10 seconds",
        "max_runtime": "Unlimited",
        "cost_model": "Per invocation + GB-seconds",
        "best_for": "Intermittent ML traffic, scales to zero, pay-per-use",
        "example_cost": "1M requests: ~$1-5 depending on model size"
    },
])

pd.set_option('display.max_colwidth', 55)
serving_options[["service", "cost_model", "best_for", "example_cost"]]

,service,cost_model,best_for,example_cost
0,EC2 (Virtual Machine),Per hour (whether idle or not),"High-traffic APIs, models needing GPU, full control",t3.medium: ~$30/month
1,Lambda (Serverless),Per invocation + duration (free tier: 1M requests/m...,"Low-traffic APIs, lightweight models (<250MB), even...",1M requests/month: ~$0.20
2,SageMaker Endpoint,Per hour per instance type,"Production ML serving, auto-scaling, A/B testing bu...",ml.t2.medium: ~$50/month
3,SageMaker Serverless,Per invocation + GB-seconds,"Intermittent ML traffic, scales to zero, pay-per-use",1M requests: ~$1-5 depending on model size


## 5. Decision Framework — Which Service to Choose

In [5]:
decision_framework = """
CLOUD ML SERVING DECISION FRAMEWORK
=====================================

Step 1: How many requests per day?
  < 1,000/day    -> Lambda (serverless, cheap, scales to zero)
  1,000-100K/day -> EC2 t3.medium or SageMaker Serverless
  > 100K/day     -> EC2 with auto-scaling or SageMaker Endpoint

Step 2: Does the model need a GPU?
  Yes -> EC2 g4dn.xlarge or SageMaker ml.g4dn.xlarge
  No  -> Lambda, EC2 t3/t4g, SageMaker ml.t2/m5

Step 3: What is the model size?
  < 250MB -> Lambda (fits in deployment package)
  250MB-5GB -> EC2 or SageMaker (Lambda has size limits)
  > 5GB   -> EC2 or SageMaker with S3 model loading

Step 4: What is the latency requirement?
  < 100ms -> EC2 with model loaded in memory (always warm)
  < 1s    -> SageMaker real-time endpoint
  1-10s OK -> SageMaker Serverless (has cold starts)
  Minutes OK -> Lambda or SageMaker Batch Transform

Step 5: Is traffic predictable?
  Yes (steady) -> EC2 (reserved instance saves 40-70%)
  No (spiky)   -> Lambda or SageMaker Serverless (pay per use)

OUR FASTAPI ML SERVICE (Day 65) ON AWS:
  Model size: small (<100MB)
  Traffic: moderate (< 10K/day initially)
  Latency: need < 500ms
  -> Best fit: EC2 t3.small ($15/month) or Lambda ($0.20/month)
  -> Start with Lambda, migrate to EC2 when traffic justifies cost
"""
print(decision_framework)


CLOUD ML SERVING DECISION FRAMEWORK

Step 1: How many requests per day?
  < 1,000/day    -> Lambda (serverless, cheap, scales to zero)
  1,000-100K/day -> EC2 t3.medium or SageMaker Serverless
  > 100K/day     -> EC2 with auto-scaling or SageMaker Endpoint

Step 2: Does the model need a GPU?
  Yes -> EC2 g4dn.xlarge or SageMaker ml.g4dn.xlarge
  No  -> Lambda, EC2 t3/t4g, SageMaker ml.t2/m5

Step 3: What is the model size?
  < 250MB -> Lambda (fits in deployment package)
  250MB-5GB -> EC2 or SageMaker (Lambda has size limits)
  > 5GB   -> EC2 or SageMaker with S3 model loading

Step 4: What is the latency requirement?
  < 100ms -> EC2 with model loaded in memory (always warm)
  < 1s    -> SageMaker real-time endpoint
  1-10s OK -> SageMaker Serverless (has cold starts)
  Minutes OK -> Lambda or SageMaker Batch Transform

Step 5: Is traffic predictable?
  Yes (steady) -> EC2 (reserved instance saves 40-70%)
  No (spiky)   -> Lambda or SageMaker Serverless (pay per use)

OUR FASTAPI ML

## 6. Cost Estimation Calculator

In [6]:
def estimate_monthly_cost(
    service: str,
    requests_per_day: int,
    avg_duration_ms: float,
    model_size_gb: float,
    data_storage_gb: float
) -> dict:
    """Estimate monthly AWS cost for ML serving."""
    
    requests_per_month = requests_per_day * 30
    
    if service == "lambda":
        # Lambda pricing (us-east-1)
        free_requests = 1_000_000
        free_gb_seconds = 400_000
        
        billable_requests = max(0, requests_per_month - free_requests)
        request_cost = billable_requests * 0.0000002  # $0.20 per 1M
        
        gb_seconds = requests_per_month * (avg_duration_ms / 1000) * 0.512  # 512MB default
        billable_gb_seconds = max(0, gb_seconds - free_gb_seconds)
        compute_cost = billable_gb_seconds * 0.0000166667
        
        storage_cost = data_storage_gb * 0.023
        total = request_cost + compute_cost + storage_cost
        
        return {
            "service": "AWS Lambda",
            "requests_per_month": f"{requests_per_month:,}",
            "request_cost": f"${request_cost:.4f}",
            "compute_cost": f"${compute_cost:.4f}",
            "storage_cost": f"${storage_cost:.2f}",
            "total_monthly": f"${total:.2f}"
        }
    
    elif service == "ec2_t3_medium":
        # EC2 t3.medium on-demand (us-east-1)
        hourly_rate = 0.0416
        instance_cost = hourly_rate * 24 * 30
        storage_cost = data_storage_gb * 0.023
        # EBS storage for EC2 (20GB root volume)
        ebs_cost = 20 * 0.08
        total = instance_cost + storage_cost + ebs_cost
        
        return {
            "service": "EC2 t3.medium",
            "instance_cost": f"${instance_cost:.2f}/month",
            "storage_cost": f"${storage_cost:.2f}",
            "ebs_cost": f"${ebs_cost:.2f}",
            "total_monthly": f"${total:.2f}"
        }
    
    elif service == "sagemaker_serverless":
        # SageMaker Serverless pricing
        request_cost = requests_per_month * 0.00002  # $0.02 per 1K invocations approx
        compute_cost = requests_per_month * (avg_duration_ms / 1000) * model_size_gb * 0.00001667
        storage_cost = data_storage_gb * 0.023
        total = request_cost + compute_cost + storage_cost
        
        return {
            "service": "SageMaker Serverless",
            "requests_per_month": f"{requests_per_month:,}",
            "request_cost": f"${request_cost:.2f}",
            "compute_cost": f"${compute_cost:.4f}",
            "storage_cost": f"${storage_cost:.2f}",
            "total_monthly": f"${total:.2f}"
        }

# scenario 1: low traffic startup (1K requests/day)
print("=== Scenario 1: Low Traffic (1,000 requests/day) ===")
for svc in ["lambda", "ec2_t3_medium", "sagemaker_serverless"]:
    result = estimate_monthly_cost(svc, 1_000, 200, 0.1, 10)
    print(f"\n{result['service']}:")
    for k, v in result.items():
        if k != "service":
            print(f"  {k}: {v}")

print("\n=== Scenario 2: Medium Traffic (10,000 requests/day) ===")
for svc in ["lambda", "ec2_t3_medium", "sagemaker_serverless"]:
    result = estimate_monthly_cost(svc, 10_000, 200, 0.1, 10)
    print(f"\n{result['service']}: Total = {result['total_monthly']}/month")

=== Scenario 1: Low Traffic (1,000 requests/day) ===

AWS Lambda:
  requests_per_month: 30,000
  request_cost: $0.0000
  compute_cost: $0.0000
  storage_cost: $0.23
  total_monthly: $0.23

EC2 t3.medium:
  instance_cost: $29.95/month
  storage_cost: $0.23
  ebs_cost: $1.60
  total_monthly: $31.78

SageMaker Serverless:
  requests_per_month: 30,000
  request_cost: $0.60
  compute_cost: $0.0100
  storage_cost: $0.23
  total_monthly: $0.84

=== Scenario 2: Medium Traffic (10,000 requests/day) ===

AWS Lambda: Total = $0.23/month

EC2 t3.medium: Total = $31.78/month

SageMaker Serverless: Total = $6.33/month


## 7. SageMaker Overview — The Managed ML Platform

In [7]:
sagemaker_overview = """
AWS SAGEMAKER -- Managed End-to-End ML Platform
=================================================

SAGEMAKER IS NOT JUST FOR INFERENCE -- it covers the full ML lifecycle:

1. SAGEMAKER STUDIO
   Cloud-based Jupyter environment with ML-optimised instance types.
   Replaces your local notebook for large-scale training.

2. SAGEMAKER TRAINING JOBS
   Run training on managed infrastructure -- no need to manage EC2 yourself.
   Supports distributed training across multiple GPU instances.
   Usage: sm.create_training_job(AlgorithmSpecification, InputDataConfig, ...)

3. SAGEMAKER EXPERIMENTS (similar to MLflow Day 66)
   Built-in experiment tracking -- logs params, metrics, artifacts.
   Integrated with SageMaker Studio UI.

4. SAGEMAKER MODEL REGISTRY
   Same concept as MLflow model registry (None/Staging/Production/Archived).
   Integrated with SageMaker deployment pipeline.

5. SAGEMAKER ENDPOINTS (Real-time inference)
   Deploy a model to a managed REST API in minutes.
   sm.create_endpoint() -> auto-scales, has A/B testing, blue/green deploys.
   Usage: predictor = sm.deploy(initial_instance_count=1, instance_type='ml.t2.medium')

6. SAGEMAKER BATCH TRANSFORM
   Run inference on a large dataset stored in S3 -- no real-time API needed.
   Cost-effective for offline scoring: process 1M rows overnight, pay only for runtime.

7. SAGEMAKER JUMPSTART
   Pre-trained foundation models (Llama, Mistral, Stable Diffusion) deployable in clicks.
   One-click fine-tuning for common tasks.

THE KEY SAGEMAKER ADVANTAGE:
   Takes our Day 65 FastAPI service and adds:
   - Auto-scaling (handles traffic spikes automatically)
   - A/B testing (route 10% to new model, 90% to old)
   - Built-in monitoring (data drift, model quality -- like Day 69's Evidently AI)
   - No infrastructure management
   
   Trade-off: costs more than self-managed EC2 + Docker.
"""
print(sagemaker_overview)


AWS SAGEMAKER -- Managed End-to-End ML Platform

SAGEMAKER IS NOT JUST FOR INFERENCE -- it covers the full ML lifecycle:

1. SAGEMAKER STUDIO
   Cloud-based Jupyter environment with ML-optimised instance types.
   Replaces your local notebook for large-scale training.

2. SAGEMAKER TRAINING JOBS
   Run training on managed infrastructure -- no need to manage EC2 yourself.
   Supports distributed training across multiple GPU instances.
   Usage: sm.create_training_job(AlgorithmSpecification, InputDataConfig, ...)

3. SAGEMAKER EXPERIMENTS (similar to MLflow Day 66)
   Built-in experiment tracking -- logs params, metrics, artifacts.
   Integrated with SageMaker Studio UI.

4. SAGEMAKER MODEL REGISTRY
   Same concept as MLflow model registry (None/Staging/Production/Archived).
   Integrated with SageMaker deployment pipeline.

5. SAGEMAKER ENDPOINTS (Real-time inference)
   Deploy a model to a managed REST API in minutes.
   sm.create_endpoint() -> auto-scales, has A/B testing, blue/green

## 8. Summary — What We Covered Today

| Service | Type | Cost model | Best for |
|---------|------|------------|----------|
| S3 | Object storage | $0.023/GB/month | Datasets, model artifacts, logs |
| EC2 | Virtual machine | Per hour (always on) | High traffic, GPU workloads, full control |
| Lambda | Serverless | Per invocation (free tier 1M/month) | Low traffic, lightweight models <250MB |
| SageMaker Endpoint | Managed ML inference | Per hour per instance | Production ML, auto-scaling, A/B testing |
| SageMaker Serverless | Serverless ML | Per invocation + GB-seconds | Intermittent traffic, pay-per-use |
| SageMaker Batch | Offline inference | Per hour (only during job) | Large dataset scoring, overnight jobs |

**Cost comparison at 1,000 requests/day:**
- Lambda:               $0.23/month (almost free at low traffic)
- SageMaker Serverless: $0.84/month
- EC2 t3.medium:        $31.78/month (same cost whether 1 or 100K requests/day)

**Key decision rules:**
1. < 1K requests/day → Lambda
2. 1K-100K/day → SageMaker Serverless or EC2
3. > 100K/day → EC2 with auto-scaling or SageMaker Endpoint
4. Needs GPU → EC2 g4dn or SageMaker GPU instances
5. Batch scoring → SageMaker Batch Transform